# Swarms

Read the [guide](README.md). Install the notebooks, multiagent and partner groups for the full tier. All model calls below replay offline; local A2A transports still use real child processes.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["MEM0_TELEMETRY"] = "false"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_handoffs

Predict the result, then run the [script](examples/01_handoffs.py).


In [ ]:
"""Handoffs and shared context: Nimbus Tier 4 lesson.

Run: uv run python modules/16_swarms/examples/01_handoffs.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

from common.coordination import report
from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.swarm_lessons import itinerary_swarm


def main() -> None:
    _, cfg = lesson_settings()
    swarm = itinerary_swarm(cfg)
    # KEY IDEA: the model selects handoff_to_agent; the runtime checks its limits.
    result = swarm("Design a public Porto itinerary. Do not book anything.")
    report(result, cfg)
    complete()


run_lesson(main, "modules/16_swarms/examples/01_handoffs.py")

## 02_swarm_limits

Predict the result, then run the [script](examples/02_swarm_limits.py).


In [ ]:
"""Handoff, repetition and timeout boundaries: Nimbus Tier 4 lesson.

Run: uv run python modules/16_swarms/examples/02_swarm_limits.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

import asyncio

from common.config import load_settings
from common.coordination import report
from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.swarm_lessons import itinerary_swarm, timeout_probe


def main() -> None:
    _, cfg = lesson_settings()
    for limits in ({"max_handoffs": 2}, {"max_iterations": 2}, {}):
        result = itinerary_swarm(cfg, loop=True, **limits)("Hand off the draft")
        print("Limits:", limits or "repetition detection")
        report(result, cfg)
    # KEY IDEA: inject a slow local tool to make cancellation reproducible, even live.
    for overall in (False, True):
        result, closed = asyncio.run(timeout_probe(load_settings(provider="mock"), overall=overall))
        status = result.status.value if result else "host deadline exceeded"
        print("Overall timeout:", overall, "status:", status, "closed:", bool(closed))
    complete()


run_lesson(main, "modules/16_swarms/examples/02_swarm_limits.py")

## 03_swarm_results

Predict the result, then run the [script](examples/03_swarm_results.py).


In [ ]:
"""Inspect node outcomes: Nimbus Tier 4 lesson.

Run: uv run python modules/16_swarms/examples/03_swarm_results.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

from common.coordination import report
from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.swarm_lessons import itinerary_swarm


def main() -> None:
    _, cfg = lesson_settings()
    for loop in (False, True):
        result = itinerary_swarm(cfg, loop=loop)("Plan a synthetic Porto draft")
        report(result, cfg)
        # KEY IDEA: inspect status and visited nodes before accepting reassuring prose.
        for name, node in result.results.items():
            print(name, node.status.value, str(node.result)[:250])
    complete()


run_lesson(main, "modules/16_swarms/examples/03_swarm_results.py")

## 04_itinerary_team

Predict the result, then run the [script](examples/04_itinerary_team.py).


In [ ]:
"""Four-role itinerary team: Nimbus Tier 4 lesson.

Run: uv run python modules/16_swarms/examples/04_itinerary_team.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

import json

from common.coordination import report
from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.swarm_lessons import itinerary_swarm
from common.trip_planning import TripRequest, catalog


def main() -> None:
    _, cfg = lesson_settings()
    request = TripRequest()
    evidence = {role: catalog(role, request) for role in ("flight", "hotel", "policy")}
    task = "Design a two-day Porto itinerary from this public evidence: " + json.dumps(evidence)
    result = itinerary_swarm(cfg)(task)
    report(result, cfg)
    print("Reviewer:", str(result.results.get("reviewer")))
    print("This creative handoff exercise does not authorize spending.")
    complete()


run_lesson(main, "modules/16_swarms/examples/04_itinerary_team.py")

## itinerary_swarm

Predict the result, then run the [script](use_case/itinerary_swarm.py).


In [ ]:
"""Itinerary design swarm: Nimbus Tier 4 lesson.

Run: uv run python modules/16_swarms/use_case/itinerary_swarm.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

import json

from common.coordination import report
from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.swarm_lessons import itinerary_swarm
from common.trip_planning import TripRequest, catalog


def main() -> None:
    _, cfg = lesson_settings()
    request = TripRequest()
    evidence = {role: catalog(role, request) for role in ("flight", "hotel", "policy")}
    task = "Design a two-day Porto itinerary from this public evidence: " + json.dumps(evidence)
    result = itinerary_swarm(cfg)(task)
    report(result, cfg)
    print("Reviewer:", str(result.results.get("reviewer")))
    print("This creative handoff exercise does not authorize spending.")
    complete()


run_lesson(main, "modules/16_swarms/use_case/itinerary_swarm.py")

## Practice

Complete the [exercises](README.md#exercises) and [quiz](quiz.md). Module 17 defaults to rejection; use the standalone --decision ask command to review and approve a draft.
